# VRDet: Stage 02/03: Features and Regime Labels

1. Build causal features from 15-minute NIFTY bars (+ India VIX, USD/INR, Brent).
2. Fit regime labelers (**HMM** primary, GMM and GARCH for comparison) on 2015–2021 only.
3. Sanity-check labels against known market shocks, inspect episodes, compare live vs hindsight decoding.
4. Save labels and the fitted HMM for the modelling stage and the app.

Code lives in `src/vrdet/`; this notebook only orchestrates and visualises.

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/agentsmith1337/VolatilityRegimeDetection.git'
DRIVE_FOLDER_URL = ''  # Colab only: share link of the Drive folder holding the raw parquet files

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = Path('/content/VolatilityRegimeDetection')
    if REPO.exists():
        subprocess.run(['git', '-C', str(REPO), 'pull', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch', 'gdown', 'hmmlearn'], check=True)
    RAW = Path('/content/vrdet_data/raw')
    if not any(RAW.rglob('*.parquet')):
        import gdown
        RAW.mkdir(parents=True, exist_ok=True)
        gdown.download_folder(DRIVE_FOLDER_URL, output=str(RAW), quiet=True)
    OUT = Path('/content/vrdet_data/processed')
else:
    REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    OUT = REPO / 'data' / 'processed'

sys.path.insert(0, str(REPO / 'src'))
OUT.mkdir(parents=True, exist_ok=True)
print('repo:', REPO, '| colab:', IN_COLAB)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score

from vrdet.data import load_raw
from vrdet.features import build_features, seasonal_shape
from vrdet.labels import (HMMLabeler, GMMLabeler, GARCHLabeler, STATE_NAMES,
                          event_report, regime_episodes)

pd.set_option('display.width', 200)
COLORS = {0: 'tab:green', 1: 'gold', 2: 'tab:red'}
TRAIN_END = '2021-12-31'   # labelers are fitted on data up to here only

## 1. Data validation and features
Every intraday session is checked against the official daily candles (`vrdet.quality`): a session is valid if it is
not flat, has at least 13 bars, and reproduces at least 2 of the official daily high / low / close within 1%.
Invalid NIFTY sessions are dropped; invalid India VIX sessions become missing.

In [ ]:
from vrdet.quality import session_report, yearly_summary
raw = load_raw()
for name in ('NIFTY', 'INDIAVIX'):
    rep = session_report(raw[f'{name}_15m'], raw[f'{name}_1d'])
    print(f'{name} 15m: {(~rep.valid).sum()} invalid sessions')
    display(yearly_summary(rep))

In [ ]:
nifty, vix = raw['NIFTY_15m'], raw['INDIAVIX_15m']
daily = {'usdinr': raw['USDINR_1d'], 'brent': raw['BRENT_1d']}

f = build_features(nifty, vix, daily, nifty_daily=raw['NIFTY_1d'], vix_daily=raw['INDIAVIX_1d'])
print(f.shape, f.index[0], '->', f.index[-1])
f.describe().T[['count', 'mean', 'std', 'min', 'max']].round(4)

### Intraday seasonality
Raw 15-minute volatility is U-shaped across the session. Without removing it, a regime model would simply learn
"09:15 = Chaotic, 12:30 = Quiet". `z` and `range_ds` divide by a causal time-of-day profile.

In [ ]:
r = f['ret']
fig, ax = plt.subplots(1, 2, figsize=(15, 3.5))
r.abs().groupby(r.index.time).mean().plot(kind='bar', ax=ax[0], title='Mean |return| by bar time (raw)')
f['z'].abs().groupby(f.index.time).mean().plot(kind='bar', ax=ax[1], title='Mean |z| by bar time (deseasonalised)')
plt.tight_layout(); plt.show()

## 2. HMM labeler (primary)
3-state Gaussian HMM (`hmmlearn`, diagonal covariance) on `[z, log range_ds]` (standardised), with a sticky
Dirichlet prior on self-transitions, states ordered by volatility, and a minimum regime length of 4 bars (1 hour).
Fitted on 2015–2021; decoded over the full history.

- **Hindsight labels** = Viterbi path (`hmmlearn` `predict`); uses the whole sequence, so it is the *target*.
- **Live labels** = forward-filtered probabilities P(state_t | data up to t) (Rabiner, 1989). `hmmlearn` only
  provides smoothed probabilities (which use future bars), so the filter is a short function in `vrdet.labels`.

In [ ]:
hmm_lab = HMMLabeler().fit(f, end=TRAIN_END)
import hmmlearn
print('hmmlearn', hmmlearn.__version__, '| log-likelihood:', round(hmm_lab.loglik_, 1))
display(hmm_lab.summary())
pd.DataFrame(hmm_lab.transmat_, index=STATE_NAMES, columns=STATE_NAMES).round(4)

In [ ]:
labels = hmm_lab.hindsight(f)            # target: hindsight (Viterbi) labels
live = hmm_lab.live(f)                   # benchmark: real-time filtered labels
p_live = hmm_lab.filtered_proba(f)

print('Regime share (hindsight):')
print(labels.map(dict(enumerate(STATE_NAMES))).value_counts(normalize=True).round(3))

### Sanity check: known market shocks should be Chaotic

In [ ]:
event_report(labels)

### Regimes over time

In [ ]:
def plot_regimes(start=None, end=None, title=''):
    c = nifty['close'].loc[start:end]
    lab = labels.reindex(c.index)
    x = np.arange(len(c))  # bar index -> no overnight/weekend gaps
    fig, ax = plt.subplots(figsize=(15, 4))
    ax.plot(x, c.to_numpy(), color='black', lw=0.6)
    lo, hi = c.min(), c.max()
    for k, col in COLORS.items():
        ax.fill_between(x, lo, hi, where=(lab == k).to_numpy(), color=col, alpha=0.25, step='mid', lw=0,
                        label=STATE_NAMES[k])
    ticks = np.linspace(0, len(c) - 1, 8).astype(int)
    ax.set_xticks(ticks, [c.index[i].strftime('%Y-%m-%d') for i in ticks])
    ax.set_title(title); ax.legend(loc='upper left'); plt.tight_layout(); plt.show()

plot_regimes(title='NIFTY 50, 15m: HMM regimes (full history)')
plot_regimes('2020-02-15', '2020-04-30', 'COVID crash')
plot_regimes('2024-05-27', '2024-06-14', 'General election results (4 Jun 2024)')
plot_regimes('2025-03-25', '2025-04-20', 'US tariff shock (Apr 2025)')

### Chaotic episodes

In [ ]:
ep = regime_episodes(labels, state=2)
ep['hours'] = ep['n_bars'] / 4
print(f'{len(ep)} Chaotic episodes; length (hours):')
print(ep['hours'].describe().round(2))

fig, ax = plt.subplots(1, 2, figsize=(15, 3.5))
(labels == 2).groupby(labels.index.year).mean().plot(kind='bar', ax=ax[0], color='tab:red',
                                                     title='Share of Chaotic bars by year')
ep.groupby(ep['start'].dt.year).size().plot(kind='bar', ax=ax[1], title='Chaotic episodes by year')
plt.tight_layout(); plt.show()
ep.sort_values('n_bars', ascending=False).head(10)

### Live (filtered) vs hindsight labels
The live HMM filter is what can be computed in real time. This is the key benchmark the ML classifiers must beat
(formal metrics in Stage 04).

In [ ]:
idx = labels.index
ct = pd.crosstab(labels.map(dict(enumerate(STATE_NAMES))).rename('hindsight'),
                 live.reindex(idx).map(dict(enumerate(STATE_NAMES))).rename('live'), normalize='index').round(3)
print('agreement:', round((labels == live.reindex(idx)).mean(), 3))
ct

## 3. Alternative labelers: GMM and GARCH
Same training span. GMM ignores time ordering; GARCH bins one-step-ahead conditional volatility
(30th / 94th training percentiles).

In [ ]:
others = {'GMM': GMMLabeler().fit(f, end=TRAIN_END), 'GARCH': GARCHLabeler().fit(f, end=TRAIN_END)}
all_labels = {'HMM': labels, **{k: v.hindsight(f) for k, v in others.items()}}
all_live = {'HMM': live, **{k: v.live(f) for k, v in others.items()}}

rows = []
for name, lab in all_labels.items():
    ev = event_report(lab)
    rows.append({'labeler': name,
                 **{s: round((lab == k).mean(), 3) for k, s in enumerate(STATE_NAMES)},
                 'chaotic_episodes': len(regime_episodes(lab)),
                 'event_chaotic_mean': round(ev['Chaotic'].mean(), 2),
                 'event_chaotic_min': round(ev['Chaotic'].min(), 2),
                 'live_vs_hindsight': round((lab == all_live[name].reindex(lab.index)).mean(), 3)})
display(pd.DataFrame(rows).set_index('labeler'))

common = all_labels['HMM'].index.intersection(all_labels['GARCH'].index)
names = list(all_labels)
kappa = pd.DataFrame([[cohen_kappa_score(all_labels[a].reindex(common), all_labels[b].reindex(common))
                       for b in names] for a in names], index=names, columns=names).round(3)
kappa

**Reading the comparison**
- *event_chaotic_mean*: share of bars labelled Chaotic on known shock sessions; higher is better.
- *live_vs_hindsight*: if close to 1, the labeler's real-time output already equals its target, so there is nothing
  for a classifier to learn (GARCH). If too low, the target is unreachable in real time.
- The HMM is the primary target; GMM/GARCH labels are kept for robustness checks.

## 4. Save

In [ ]:
out = f.join(labels.rename('regime')).join(live.rename('regime_live_hmm'))
out = out.join(p_live.add_prefix('p_live_'))
for k, v in all_labels.items():
    if k != 'HMM':
        out[f'regime_{k.lower()}'] = v
out.to_parquet(OUT / 'features_labels_15m.parquet')

(REPO / 'models').mkdir(exist_ok=True)
(REPO / 'models' / 'hmm_labeler.json').write_text(hmm_lab.to_json())
print('saved', OUT / 'features_labels_15m.parquet', out.shape, '| models/hmm_labeler.json')